In [ ]:
import numpy as np
from env import RaceEnviroment
import matplotlib.pyplot as plt

# Funktionen

In [ ]:
def make_Q(race):
    coords = np.argwhere(np.isin(race.track, [1,2,3]))
    n_states = len(coords)
    n_actions = len(race.actions)
    Q = np.zeros((n_states, n_actions))
    return Q, coords

def make_Pi(race):
    n_states = np.sum(np.isin(race.track, [1, 2, 3]))
    n_actions = len(race.actions)

    Pi = np.zeros((n_states, n_actions), dtype=float)
    return Pi

def state_to_index(S, coords):
    return np.where((coords[:,0] == S[0]) & (coords[:,1] == S[1]))[0][0]

def choose_A(S, Q, coords, epsilon):
    state_index = state_to_index(S, coords)
    if np.random.rand() < epsilon:
        return np.random.choice(Q.shape[1])
    else:
        return np.argmax(Q[state_index, :])
    
def update_pi(Pi, Q, epsilon):
    n_states, n_actions = Q.shape

    for s in range(n_states):
        # beste Aktion
        best_a = np.argmax(Q[s])

        # alle Aktionen erstmal gleich verteilen (ε-Exploration)
        Pi[s, :] = epsilon / n_actions

        # beste Aktion bekommt (1 - ε) extra
        Pi[s, best_a] += 1 - epsilon
    
    return Pi

# Main

In [ ]:
race = RaceEnviroment("a", 5, 0, -1)

gamma = 1
epsilon = 0.1
required_steps = []

Q, coords = make_Q(race)
Pi = make_Pi(race)
n_states, n_actions = Q.shape
Returns = [[[] for _ in range(n_actions)] for _ in range(n_states)]
episodes = 30000

In [ ]:
for ep in range(episodes):
    race.reset()
    episode = []
    S = (race.y, race.x)
    
    # komplette Episode generieren
    while True:
        A = choose_A(S, Q, coords, epsilon)
        R, next_S = race.step(A+1)
        episode.append((S, A, R))
        S = next_S
        if R == 0:  # Ziel erreicht
            break
    
    # Rückwärts G berechnen
    G = 0
    visited = set()
    for S_t, A_t, R_t in reversed(episode):
        G = gamma * G + R_t
        S_idx = state_to_index(S_t, coords)
        if (S_idx, A_t) not in visited:  # first-visit
            Returns[S_idx][A_t].append(G)
            Q[S_idx, A_t] = np.mean(Returns[S_idx][A_t])
            visited.add((S_idx, A_t))
    
    Pi = update_pi(Pi, Q, epsilon)
    required_steps.append(len(episode))

In [ ]:
plt.figure()
plt.bar(range(len(required_steps)), required_steps)
plt.xlabel("Episode")
plt.ylabel("Required Steps")
plt.title("Steps per Episode")
plt.ylim(0,500)
plt.show()